## SAM 3 video segmentation on robot manipulation data — ALL EPISODES (batch)

Batch sibling of [`sam3_video_robot_data_single_episode.ipynb`](sam3_video_robot_data_single_episode.ipynb). Pick **one task** and **one prompt** at the top, then run all episodes in that task end-to-end (stage → prompt → propagate → save masks → close session, per episode).

Use the single-episode notebook for tasks where text/point can't generalize across episodes (interactive box/point picker per episode).

Output: per-episode `<episode>/mask/00000_mask.png … NNNNN_mask.png` — 3-channel RGB binary PNGs at the head-cam JPEG resolution, drop-in as an extra camera for ACT-style policies.

In [15]:
!nvidia-smi

Thu May  7 18:38:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.119.02             Driver Version: 580.119.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:01:00.0  On |                  N/A |
|  0%   46C    P1             67W /  600W |    8276MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Imports & build predictor

In [16]:
import copy
import glob
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import sam3
import torch
from PIL import Image
from sam3.model_builder import build_sam3_video_predictor
from sam3.visualization_utils import (
    load_frame,
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
)

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12


def propagate_in_video(predictor, session_id, start_frame_index=0):
    """Propagate forward+backward from a given frame.

    Passing start_frame_index=0 explicitly is required for the point-prompt
    (instance-tracking) path: that path doesn't populate `previous_stages_out`,
    so without an explicit start the predictor raises
    'No prompts are received on any frames'. Text/box prompts work either way.
    """
    outputs_per_frame = {}
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
            start_frame_index=start_frame_index,
        )
    ):
        outputs_per_frame[response["frame_index"]] = response["outputs"]
    return outputs_per_frame

In [17]:
gpus_to_use = list(range(torch.cuda.device_count()))
predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)

INFO 2026-05-07 18:38:08,992 468172 sam3_video_predictor.py: 109: using the following GPU IDs: [0]
INFO 2026-05-07 18:38:08,993 468172 sam3_video_predictor.py: 125: 


	*** START loading model on all ranks ***


INFO 2026-05-07 18:38:08,993 468172 sam3_video_predictor.py: 127: loading model on rank=0 with world_size=1 -- this could take a while ...
INFO 2026-05-07 18:38:11,524 468172 sam3_video_base.py: 348: setting max_num_objects=10000 and num_obj_for_compile=16
INFO 2026-05-07 18:38:13,602 468172 sam3_video_predictor.py: 129: loading model on rank=0 with world_size=1 -- DONE locally
INFO 2026-05-07 18:38:13,603 468172 sam3_video_predictor.py: 140: 


	*** DONE loading model on all ranks ***




### Configure: pick task + prompt

Edit the cell below. Choose `PROMPT_MODE = "text"` for chip_pick (concept-based, generalizes), or `PROMPT_MODE = "point"` for kit_pick / mask_pick where the head cam is fixed and the target starts in roughly the same workspace zone every episode.

In [ ]:
DATASET_ROOT     = os.path.expanduser("~/datasets/manipulation_dataset_v0.1")
HEAD_CAM_SUFFIX  = "_color_1.jpg"
TASK             = "chip_pick"        # <-- edit me  (chip_pick | kit_pick | mask_pick)

# Pick exactly one prompt mode. For "point", populate POINTS_PIXEL via the picker cell below.
PROMPT_MODE      = "text"             # "text"  | "point"
TEXT_PROMPT      = "bag of chips"     # used if PROMPT_MODE == "text"
OBJ_ID           = 1                  # used if PROMPT_MODE == "point"
POINTS_PIXEL     = []                 # populated by the picker cell; reused for every episode

task_dir = os.path.join(DATASET_ROOT, TASK)
assert os.path.isdir(task_dir), f"missing task dir: {task_dir}"
episodes = sorted(
    d for d in os.listdir(task_dir)
    if d.startswith("episode_") and os.path.isdir(os.path.join(task_dir, d))
)
print(f"task: {TASK}    mode: {PROMPT_MODE}    episodes: {len(episodes)}  ({episodes[0]} ... {episodes[-1]})")
if PROMPT_MODE == "text":
    print(f"text prompt: {TEXT_PROMPT!r}")
elif PROMPT_MODE == "point":
    print(f"obj_id: {OBJ_ID}    POINTS_PIXEL ({len(POINTS_PIXEL)}): {POINTS_PIXEL}")
else:
    raise ValueError(f"unknown PROMPT_MODE: {PROMPT_MODE!r}")

### Per-episode helpers

These are the same primitives the single-episode notebook uses, hoisted into functions so the batch loop can call them per episode.

In [ ]:
def stage_head_cam(episode_dir):
    """Build (or refresh) head_cam_frames/00000.jpg ... pointing at *_color_1.jpg files.
    Returns (stage_dir, head_frames_src). Idempotent: clears stale links first."""
    colors_dir = os.path.join(episode_dir, "colors")
    head_frames_src = sorted(glob.glob(os.path.join(colors_dir, f"*{HEAD_CAM_SUFFIX}")))
    if not head_frames_src:
        raise RuntimeError(f"no *{HEAD_CAM_SUFFIX} files in {colors_dir}")
    stage_dir = os.path.join(episode_dir, "head_cam_frames")
    os.makedirs(stage_dir, exist_ok=True)
    for old in glob.glob(os.path.join(stage_dir, "*.jpg")):
        if os.path.islink(old) or os.path.isfile(old):
            os.unlink(old)
    for i, src in enumerate(head_frames_src):
        os.symlink(os.path.abspath(src), os.path.join(stage_dir, f"{i:05d}.jpg"))
    return stage_dir, head_frames_src


def apply_prompt(predictor, session_id, head_frames_src):
    """Apply the configured PROMPT_MODE to frame 0. Returns the response['outputs'].
    For PROMPT_MODE='point', uses every (x, y, label) tuple in POINTS_PIXEL — left-clicks
    are positive (1), right-clicks are negative (0). Coordinates are converted to the
    per-episode image size (so it still works if any episode has a slightly different
    resolution)."""
    if PROMPT_MODE == "text":
        resp = predictor.handle_request(dict(
            type="add_prompt", session_id=session_id,
            frame_index=0, text=TEXT_PROMPT,
        ))
    elif PROMPT_MODE == "point":
        if not POINTS_PIXEL:
            raise RuntimeError("PROMPT_MODE='point' but POINTS_PIXEL is empty — run the picker cell first")
        with Image.open(head_frames_src[0]) as im:
            W_img, H_img = im.size
        xs_rel = [x / W_img for (x, _, _) in POINTS_PIXEL]
        ys_rel = [y / H_img for (_, y, _) in POINTS_PIXEL]
        labels = [lab     for (_, _, lab) in POINTS_PIXEL]
        points = torch.tensor(list(zip(xs_rel, ys_rel)), dtype=torch.float32)
        point_labels = torch.tensor(labels, dtype=torch.int32)
        resp = predictor.handle_request(dict(
            type="add_prompt", session_id=session_id,
            frame_index=0, points=points, point_labels=point_labels,
            obj_id=OBJ_ID,
        ))
    else:
        raise ValueError(f"unknown PROMPT_MODE: {PROMPT_MODE!r}")
    return resp["outputs"]


def save_masks(outputs_per_frame_raw, head_frames_src, episode_dir):
    """Write 3-channel RGB binary masks at head-cam resolution. Always overwrites."""
    mask_dir = os.path.join(episode_dir, "mask")
    os.makedirs(mask_dir, exist_ok=True)
    for old in glob.glob(os.path.join(mask_dir, "*.png")):
        os.unlink(old)

    with Image.open(head_frames_src[0]) as im:
        target_w, target_h = im.size
    any_out = next(iter(outputs_per_frame_raw.values()))
    Hm, Wm = any_out["out_binary_masks"].shape[-2:]

    n_with_fg = 0
    for fidx in range(len(head_frames_src)):
        fg = np.zeros((Hm, Wm), dtype=bool)
        out = outputs_per_frame_raw.get(fidx)
        if out is not None:
            bin_masks = np.asarray(out["out_binary_masks"])
            if bin_masks.size:
                fg = np.any(bin_masks, axis=0)
        gray = Image.fromarray((fg.astype(np.uint8) * 255), mode="L").resize(
            (target_w, target_h), resample=Image.NEAREST
        )
        rgb = Image.merge("RGB", (gray, gray, gray))
        rgb.save(os.path.join(mask_dir, f"{fidx:05d}_mask.png"))
        if fg.any():
            n_with_fg += 1
    return len(head_frames_src), n_with_fg

### (Point mode only) Pick points on episode_0001 frame 0

Skip this cell entirely if `PROMPT_MODE = "text"`.

**Left-click** = positive point (on the target). **Right-click** = negative point (push the mask away from this spot). You can add multiple of each — usually one positive is enough but if SAM 3 grabs too much / too little, add more clicks. The list of points is reused across every episode in the batch (head cam is rigidly mounted, so pixel coordinates transfer).

**Re-running this cell clears the points** and stages the first episode again so you can start fresh.

In [ ]:
%matplotlib widget

POINTS_PIXEL = []   # list of (x, y, label); reset on re-run

_preview_ep = episodes[0]
_preview_dir = os.path.join(task_dir, _preview_ep)
_, _preview_frames = stage_head_cam(_preview_dir)

_frame0 = np.array(Image.open(_preview_frames[0]))
H_img, W_img = _frame0.shape[:2]

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(_frame0)
ax.set_title(f"{_preview_ep}  \u2014  Left-click = positive,  Right-click = negative")

def _on_click(event):
    if event.inaxes != ax:
        return
    x, y = int(event.xdata), int(event.ydata)
    if event.button == 1:
        POINTS_PIXEL.append((x, y, 1))
        ax.plot(x, y, marker="*", color="lime", markersize=18, markeredgecolor="black")
    elif event.button == 3:
        POINTS_PIXEL.append((x, y, 0))
        ax.plot(x, y, marker="X", color="red", markersize=14, markeredgecolor="black")
    else:
        return
    fig.canvas.draw_idle()
    print(f"POINTS_PIXEL ({len(POINTS_PIXEL)}): {POINTS_PIXEL}")

_cid = fig.canvas.mpl_connect("button_press_event", _on_click)
plt.show()

### Preview the prompt on the first episode

Sanity-checks your config before kicking off the long batch. Stages episode_0001's head cam, runs the prompt on frame 0, shows the overlay, and closes the session. If the overlay looks wrong, edit the config and re-run this cell.

In [ ]:
%matplotlib inline
preview_ep = episodes[0]
preview_dir = os.path.join(task_dir, preview_ep)
preview_stage_dir, preview_frames_src = stage_head_cam(preview_dir)

resp = predictor.handle_request(dict(type="start_session", resource_path=preview_stage_dir))
preview_session = resp["session_id"]
try:
    out = apply_prompt(predictor, preview_session, preview_frames_src)
    print(f"{preview_ep} frame 0: detected {len(out['out_obj_ids'])} object(s), ids={list(out['out_obj_ids'])}")
    plt.close("all")
    visualize_formatted_frame_output(
        0, preview_frames_src,
        outputs_list=[prepare_masks_for_visualization({0: out})],
        titles=[f"{preview_ep}  \u2014 frame 0  ({PROMPT_MODE})"],
        figsize=(6, 4),
    )
finally:
    predictor.handle_request(dict(type="close_session", session_id=preview_session))

### Run the full batch

Iterates every episode in the task. One failure per episode is logged but doesn't abort the run. Existing `mask/` PNGs in each episode are deleted first (always overwrite by user choice).

In [ ]:
import time

successes = []
failures = []
n_total = len(episodes)
t0 = time.time()

for i, ep in enumerate(episodes, 1):
    episode_dir = os.path.join(task_dir, ep)
    print(f"[{i:>3}/{n_total}] {ep}", end="  ", flush=True)
    session_id = None
    try:
        stage_dir, head_frames_src = stage_head_cam(episode_dir)
        resp = predictor.handle_request(dict(type="start_session", resource_path=stage_dir))
        session_id = resp["session_id"]

        out0 = apply_prompt(predictor, session_id, head_frames_src)
        if len(out0["out_obj_ids"]) == 0:
            raise RuntimeError("no detection on frame 0")

        outputs_per_frame_raw = propagate_in_video(predictor, session_id, start_frame_index=0)
        n_saved, n_with_fg = save_masks(outputs_per_frame_raw, head_frames_src, episode_dir)
        successes.append(ep)
        print(f"OK   {n_saved} masks  ({n_with_fg} non-empty)")
    except Exception as e:
        failures.append((ep, repr(e)))
        print(f"FAIL  {e!r}")
    finally:
        if session_id is not None:
            try:
                predictor.handle_request(dict(type="close_session", session_id=session_id))
            except Exception as e:
                print(f"        (close_session error: {e!r})")

elapsed = time.time() - t0
print(f"\n=== Done in {elapsed/60:.1f} min.  {len(successes)}/{n_total} succeeded ===")
if failures:
    print("Failed:")
    for ep, err in failures:
        print(f"  {ep}: {err}")

### Cleanup

Run after you're done with this task. Frees the multi-GPU process group.

In [ ]:
predictor.shutdown()